# GAT Training — Hybrid Log Analyzer

This notebook trains a Graph Attention Network (GAT) based autoencoder for
anomaly detection on graph-structured log data.

**Parameterized with Papermill** — the cell below tagged `parameters` contains
defaults that are overridden at execution time via `papermill -f configs/params.yaml`.

In [ ]:
# ── Parameters (injected by Papermill) ────────────────────
# Do NOT place inter-dependent expressions here.
dataset_path = "data/processed"
in_channels = 16
hidden_dim = 64
out_dim = 32
GAT_attention_heads = 4
dropout = 0.2
learning_rate = 0.001
batch_size = 32
epochs = 50
weight_decay = 0.0001
patience = 10
checkpoint_dir = "models"
save_every_n_epochs = 5
random_seed = 42

In [ ]:
import os, sys
sys.path.insert(0, os.path.abspath("."))

from src.utils import is_ci, is_colab, seed_everything, get_device

seed_everything(random_seed)
device = get_device()
print(f"Running on: {device}  |  CI={is_ci()}  |  Colab={is_colab()}")

In [ ]:
# ── Environment guard: mount Google Drive only in Colab ───
if is_colab():
    from google.colab import drive  # type: ignore
    drive.mount("/content/drive")
    # Optionally override checkpoint_dir to Drive path
    # checkpoint_dir = "/content/drive/MyDrive/hybrid-log-analyzer/models"

In [ ]:
import json
from pathlib import Path
import torch

# Load processed data
data_path = Path(dataset_path) / "graph_data.json"
print(f"Loading data from {data_path}")

# TODO: Replace with real data loading once preprocessing is implemented
if data_path.exists():
    with open(data_path) as f:
        graph_data = json.load(f)
    print(f"Loaded graph data with {len(graph_data.get('node_features', []))} nodes")
else:
    print(f"WARNING: {data_path} not found — using synthetic data for pipeline validation")
    graph_data = None

In [ ]:
from src.model import GraphAttentionEncoder, GraphAutoEncoder, save_checkpoint

# Build model
encoder = GraphAttentionEncoder(
    in_channels=in_channels,
    hidden_dim=hidden_dim,
    out_dim=out_dim,
    heads=GAT_attention_heads,
    dropout=dropout,
)
model = GraphAutoEncoder(encoder).to(device)
optimizer = torch.optim.Adam(model.parameters(), lr=learning_rate, weight_decay=weight_decay)
print(model)

In [ ]:
# ── Training loop (placeholder — replace with real data loading) ──
train_losses = []
val_losses = []
best_loss = float("inf")
patience_counter = 0

print(f"Training for up to {epochs} epochs (patience={patience})")

# TODO: Replace this placeholder with real training once data pipeline is complete
for epoch in range(1, epochs + 1):
    # Placeholder: simulate decreasing loss
    import random
    loss = 1.0 / (epoch + 1) + random.uniform(-0.01, 0.01)
    train_losses.append(loss)
    val_losses.append(loss * 1.1)

    if loss < best_loss:
        best_loss = loss
        patience_counter = 0
    else:
        patience_counter += 1

    if epoch % save_every_n_epochs == 0:
        ckpt_path = f"{checkpoint_dir}/gat_model.pt"
        save_checkpoint(model, optimizer, epoch, loss, {"train_loss": loss}, path=ckpt_path)
        print(f"  Epoch {epoch}/{epochs} — loss={loss:.4f} — checkpoint saved")

    if patience_counter >= patience:
        print(f"  Early stopping at epoch {epoch}")
        break

# Save final checkpoint
Path(checkpoint_dir).mkdir(parents=True, exist_ok=True)
save_checkpoint(model, optimizer, epoch, best_loss, {"train_loss": best_loss}, path=f"{checkpoint_dir}/gat_model.pt")
print(f"Training complete. Best loss: {best_loss:.4f}")

In [ ]:
# Save loss curves for evaluation stage
import json
Path("outputs").mkdir(exist_ok=True)
with open("outputs/train_history.json", "w") as f:
    json.dump({"train_losses": train_losses, "val_losses": val_losses}, f)
print("Loss history saved to outputs/train_history.json")